# 05 — All pre-set checks in one table

**Goal.** Collect every check that the plan fixed *before* the results, apply its pass/fail rule to the saved results, and save one table. A check is a test that could make a claim fail; listing them all, including the ones that fail, is what makes the results trustworthy.

**Thesis part.** All parts: Step 1 (data), Part 1 (prediction), Part 2 (association). Rules: `plan/analysis_plan.md`, sections 3–5.

**Inputs.** Saved results only: `reference/benchmark_*.csv` (from `01_benchmark`), `reports/part1_*.csv` (from `03_prediction`), `reports/part2_*.csv` (from `04_association`). Nothing heavy is recomputed.

**Outputs.** `reports/checks.csv` (read by `main.ipynb`, chapter 5).

**How to run.** Run all cells (a few seconds). Re-run after any work notebook changes.

**Status words.** PASS = the rule is met. FAIL = the rule is not met. WEAKENED = the negative-control rule says the claim is weaker. APPLIED = the rule chose the method (for example random effects). INFO = reported, no pass/fail rule.

## 1. Setup

In [1]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent
REF, R = ROOT / "reference", ROOT / "reports"
pl.Config.set_tbl_rows(60); pl.Config.set_tbl_width_chars(220); pl.Config.set_fmt_str_lengths(120)
checks = []


def add(part, check, rule, result, status, note=""):
    checks.append({"part": part, "check": check, "rule": rule, "result": result, "status": status, "note": note})

## 2. Step 1: do our data match CDC?

**Rules (plan, section 5).** (a) Record counts equal the counts printed in each user guide. (b) CDC Documentation Table 2 (births and deaths by race, sex and birth weight) is reproduced: no rate may differ by more than 0.01 per 1,000.

In [2]:
cnt = pl.read_csv(REF / "benchmark_counts.csv")
n_b, n_d = int(cnt["births_match"].sum()), int(cnt["deaths_match"].sum())
add("Step 1", "Record counts vs user guides", "births and deaths equal the guide, every cohort",
    f"births {n_b}/8, deaths {n_d}/8", "PASS" if n_d == 8 and n_b == 8 else "FAIL",
    "Failure comes from the 2020 guide, not our data: it prints 3,549,826 births (a typo); the file and NCHS agree on 3,619,826" if n_b == 7 else "")
t2 = pl.read_csv(REF / "benchmark_table2.csv")
per = t2.group_by("cohort").agg(max_diff=(pl.col("rate") - pl.col("cdc_rate")).abs().max()).sort("cohort")
ok = per.filter(pl.col("max_diff") <= 0.01)["cohort"].to_list()
bad = per.filter(pl.col("max_diff") > 0.01)["cohort"].to_list()
add("Step 1", "CDC Documentation Table 2 reproduced", "every rate within 0.01 per 1,000",
    f"exact in {len(ok)} cohorts ({min(ok)}-{max(ok)}); off in {bad}", "PASS" if not bad else "FAIL",
    "Failures come from CDC's tables, not our data: the 2016 file was re-released with birth weight for 2,493 births "
    "the guide lists as not stated; the 2017 guide copies all-races deaths into the NH White rows.")
per

cohort,max_diff
i64,f64
2016,300.0
2017,1869.53
2018,0.0
2019,0.0
2020,0.0
2021,0.0
2022,0.0
2023,0.0


**Result.** Deaths match in all 8 cohorts and births in 7; the eighth is a typo in the 2020 guide, so the count check **FAILS** the strict rule ("every cohort"). Table 2 is reproduced exactly for 2018–2023. **2016 and 2017 FAIL the strict rule.** We keep both as FAIL: the rules were fixed in advance. `01_benchmark` shows that all failures come from errors or updates in CDC's own tables, so the data are still usable.

## 3. Part 1: prediction

**Rules (plan, section 3).** (1) The best model beats model (b) in PR-AUC, and the 95% CI of the paired difference excludes 0. (2) The calibration slope of the best model is between 0.9 and 1.1. (3) No single input reaches ROC-AUC > 0.80 alone (leakage).

In [3]:
d = pl.read_csv(R / "part1_pr_auc_differences.csv").filter(pl.col("compared_with") == "b_age_race_logistic|-").row(0, named=True)
add("Part 1", "1. Best model beats age + race model", "PR-AUC difference > 0, 95% CI excludes 0",
    f"{d['pr_auc_diff']:.4f} ({d['lo']:.4f} to {d['hi']:.4f})", "PASS" if d["lo"] > 0 else "FAIL")
b = pl.read_csv(R / "part1_model_comparison.csv").filter(pl.col("is_best")).row(0, named=True)
add("Part 1", "2. Calibration slope", "0.9 <= slope <= 1.1",
    f"{b['cal_slope']:.3f} ({b['cal_slope_lo']:.3f} to {b['cal_slope_hi']:.3f}); O/E {b['observed_over_expected']:.2f}",
    "PASS" if 0.9 <= b["cal_slope"] <= 1.1 else "FAIL", "CI excludes 1: predictions slightly too spread out")
lk = pl.read_csv(R / "part1_leakage.csv")
top = lk.sort("roc_auc_alone", descending=True).row(0, named=True)
add("Part 1", "3. Single-input leakage", "no input alone has ROC-AUC > 0.80",
    f"highest {top['feature']} {top['roc_auc_alone']:.3f}", "PASS" if top["roc_auc_alone"] <= 0.80 else "FAIL",
    "The check missed timing leakage spread over precare, illb, ilp (found by importance).")
st = pl.read_csv(R / "part1_strict.csv")
sb = st.filter(pl.col("model") == "d_hgb|B_strict").row(0, named=True)
add("Part 1", "Post-hoc: model without timing items", "not pre-set; reported next to the main result",
    f"Set B PR-AUC {sb['pr_auc']:.4f} ({sb['pr_auc_lo']:.4f}-{sb['pr_auc_hi']:.4f}) vs 0.0380", "INFO",
    "The strict numbers are the more honest estimate of what is known before delivery.")
pl.DataFrame([c for c in checks if c["part"] == "Part 1"])

part,check,rule,result,status,note
str,str,str,str,str,str
"""Part 1""","""1. Best model beats age + race model""","""PR-AUC difference > 0, 95% CI excludes 0""","""0.0293 (0.0277 to 0.0308)""","""PASS""",""""""
"""Part 1""","""2. Calibration slope""","""0.9 <= slope <= 1.1""","""0.942 (0.933 to 0.950); O/E 1.03""","""PASS""","""CI excludes 1: predictions slightly too spread out"""
"""Part 1""","""3. Single-input leakage""","""no input alone has ROC-AUC > 0.80""","""highest feduc 0.626""","""PASS""","""The check missed timing leakage spread over precare, illb, ilp (found by importance)."""
"""Part 1""","""Post-hoc: model without timing items""","""not pre-set; reported next to the main result""","""Set B PR-AUC 0.0300 (0.0288-0.0314) vs 0.0380""","""INFO""","""The strict numbers are the more honest estimate of what is known before delivery."""


**Result.** All three pre-set Part 1 checks PASS. The leakage check passed but was too weak: three timing items leak a little information about preterm birth. The post-hoc strict model (PR-AUC 0.030) is reported next to the main one (0.038).

## 4. Part 2: method checks

**Rules.** (a) The modified Poisson model fitted on pattern counts equals statsmodels on single births (differences < 1e-8, synthetic data). (b) Every planned confounder set is valid under its DAG, and becomes invalid when gestational age is added.

In [4]:
m = pl.read_csv(R / "part2_method_check.csv").row(0, named=True)
add("Part 2", "Model code vs statsmodels", "coefficients, robust SE and RD agree to < 1e-8",
    f"beta {m['beta_max_diff']:.1e}, SE {m['se_max_rel_diff']:.1e}, RD {m['rd_abs_diff']:.1e}",
    "PASS" if max(m["beta_max_diff"], m["se_max_rel_diff"], m["rd_abs_diff"]) < 1e-8 else "FAIL")
dag = pl.read_csv(R / "part2_dag_check.csv")
add("Part 2", "DAG adjustment sets (back-door)", "every set valid; every set + gestational age invalid",
    f"{int(dag['valid'].sum())}/5 valid, {int((~dag['valid_with_GA']).sum())}/5 invalid with GA",
    "PASS" if dag["valid"].all() and not dag["valid_with_GA"].any() else "FAIL",
    "Valid under the drawn DAGs; the DAGs themselves are assumptions for the supervisor to review.")
dag

exposure,set,valid,valid_with_GA,unadjusted_valid
str,str,bool,bool,bool
"""hypertension""","""Age, Race, Nativity, Edu, Pay, Parity, Year, Smoke0, BMI""",true,false,false
"""diabetes""","""Age, Race, Nativity, Edu, Pay, Parity, Year, Smoke0, BMI""",true,false,false
"""bmi""","""Age, Race, Nativity, Edu, Pay, Parity, Year, Smoke0""",true,false,false
"""smoking""","""Age, Race, Nativity, Edu, Pay, Parity, Year""",true,false,false
"""interval""","""Age, Race, Nativity, Edu, Pay, Parity, Year, Smoke0, PriorLoss, PriorDead, PrevPTB""",true,false,false


## 5. Part 2: pre-set checks of the results

**Rules (plan, section 4).** (1) Direction: the per-cohort RR is on the same side of 1 as the pooled RR in at least 7 of 8 cohorts. (2) Heterogeneity: if I² > 50%, random effects is the main result. (3) Negative control: the claim is weakened if the accident-death RR excludes 1 and its CI overlaps the main RR's CI. (4) Collider demonstration: refit with gestational age, report. (5) Missing data: if missingness is linked to death (> 20% relative difference), run multiple imputation.

In [5]:
res = pl.read_csv(R / "part2_results.csv")
inf = res.filter(pl.col("outcome") == "infant")
sec = res.filter(pl.col("outcome") != "infant")
low_sec = sec.filter(pl.col("same_direction") < 7)
add("Part 2", "1. Direction in >= 7 of 8 cohorts (infant death)", ">= 7/8 cohorts on the pooled side of 1",
    f"{int((inf['same_direction'] >= 7).sum())}/{inf.height} exposure classes", "PASS" if (inf["same_direction"] >= 7).all() else "FAIL")
add("Part 2", "1b. Direction, secondary outcomes", ">= 7/8 cohorts", f"{sec.height - low_sec.height}/{sec.height} classes",
    "INFO", "Below 7: " + "; ".join(f"{r['exposure']} {r['label']} {r['outcome']} ({r['same_direction']}/8, RR {r['rr']:.2f})"
                                      for r in low_sec.iter_rows(named=True)) + " - all RRs close to 1")
hi = res.filter(pl.col("main_model") == "random")
add("Part 2", "2. Heterogeneity", "I2 > 50% -> random effects is the main result",
    f"{hi.height}/{res.height} rows with I2 > 50%", "APPLIED",
    "; ".join(f"{r['exposure']} {r['label']} {r['outcome']} (I2 {r['I2']:.0%})" for r in hi.iter_rows(named=True)) + "; RRs almost unchanged")
nc = pl.read_csv(R / "part2_negative_control.csv")
for ex, g in nc.group_by("exposure", maintain_order=True):
    w = g.filter(pl.col("check3") == "WEAKENED")
    add("Part 2", f"3. Negative control: {ex[0]}", "accident-death RR excludes 1 and overlaps the main RR -> weakened",
        "; ".join(f"{r['label']}: {r['rr_accident']:.2f} ({r['lo']:.2f}-{r['hi']:.2f})" for r in g.iter_rows(named=True)),
        "WEAKENED" if w.height else "PASS",
        ("weakened for " + ", ".join(w["label"].to_list())) if w.height else
        f"few accident deaths in the exposed ({int(g['accident_deaths_in_class'].min())}), so the test is weak")
col = pl.read_csv(R / "part2_collider.csv")
add("Part 2", "4. Collider demonstration (adjust for gestational age)", "report and explain; not a result",
    "; ".join(f"{r['exposure']} {r['label']}: {r['rr_main']:.2f} -> {r['rr_with_ga']:.2f}"
              for r in col.filter(pl.col("label").is_in(["yes", ">=40", "any", "<6 mo"])).iter_rows(named=True)),
    "INFO", "Shows why the main models do not adjust for gestational age (hypertension reverses below 1).")
ms = pl.read_csv(R / "part2_missing_check.csv")
mi = pl.read_csv(R / "part2_mi_results.csv")
add("Part 2", "5. Missing data -> multiple imputation", "relative death-rate difference > 20% triggers imputation",
    f"triggered for {int(ms['triggers_mi'].sum())}/{ms.filter(pl.col('triggers_mi').is_not_null()).height} items; "
    f"after imputing BMI, education, interval the largest RR change is {mi['change_pct'].abs().max():.1f}%",
    "APPLIED", "The plan sets no pass/fail rule for the size of the change; no conclusion changes (assumes missing at random given the model items).")
add("Part 2", "Multiple testing (Holm, 15 tests)", "sensitivity: Holm-adjusted p-values",
    f"all 15 significant; largest Holm p = {res['p_holm'].max():.1e}", "INFO")
soc = pl.read_csv(R / "part2_social_sensitivity.csv")
sm = {(r["analysis"], r["label"]): r for r in soc.filter(pl.col("exposure") == "smoking").iter_rows(named=True)}
iv = {(r["analysis"], r["label"]): r for r in soc.filter(pl.col("exposure") == "interval").iter_rows(named=True)}
add("Part 2", "Post-hoc: extra social markers (sensitivity c)", "not pre-set; tests the negative-control explanation",
    f"smoking {sm[('main confounders', 'any')]['rr']:.2f} -> {sm[('+ social markers', 'any')]['rr']:.2f} "
    f"(accident {sm[('main confounders', 'any')]['rr_accident']:.2f} -> {sm[('+ social markers', 'any')]['rr_accident']:.2f}); "
    f"interval <6 mo {iv[('main confounders', '<6 mo')]['rr']:.2f} -> {iv[('+ social markers', '<6 mo')]['rr']:.2f}",
    "INFO", "Supports social confounding for smoking only partly; does not explain the short-interval signal.")
pl.DataFrame([c for c in checks if c["part"] == "Part 2"])

part,check,rule,result,status,note
str,str,str,str,str,str
"""Part 2""","""Model code vs statsmodels""","""coefficients, robust SE and RD agree to < 1e-8""","""beta 1.3e-12, SE 4.7e-12, RD 4.3e-12""","""PASS""",""""""
"""Part 2""","""DAG adjustment sets (back-door)""","""every set valid; every set + gestational age invalid""","""5/5 valid, 5/5 invalid with GA""","""PASS""","""Valid under the drawn DAGs; the DAGs themselves are assumptions for the supervisor to review."""
"""Part 2""","""1. Direction in >= 7 of 8 cohorts (infant death)""",""">= 7/8 cohorts on the pooled side of 1""","""13/13 exposure classes""","""PASS""",""""""
"""Part 2""","""1b. Direction, secondary outcomes""",""">= 7/8 cohorts""","""23/26 classes""","""INFO""","""Below 7: bmi 25-29.9 postneonatal (5/8, RR 1.02); interval 12-17 mo neonatal (6/8, RR 1.02); interval 24-59 mo neonatal …"
"""Part 2""","""2. Heterogeneity""","""I2 > 50% -> random effects is the main result""","""5/39 rows with I2 > 50%""","""APPLIED""","""diabetes yes infant (I2 57%); bmi 35-39.9 infant (I2 66%); bmi 35-39.9 neonatal (I2 63%); smoking any infant (I2 79%); s…"
"""Part 2""","""3. Negative control: hypertension""","""accident-death RR excludes 1 and overlaps the main RR -> weakened""","""yes: 1.03 (0.76-1.40)""","""PASS""","""few accident deaths in the exposed (44), so the test is weak"""
"""Part 2""","""3. Negative control: diabetes""","""accident-death RR excludes 1 and overlaps the main RR -> weakened""","""yes: 0.80 (0.47-1.36)""","""PASS""","""few accident deaths in the exposed (14), so the test is weak"""
"""Part 2""","""3. Negative control: bmi""","""accident-death RR excludes 1 and overlaps the main RR -> weakened""","""<18.5: 1.27 (0.99-1.62); 25-29.9: 1.02 (0.90-1.16); 30-34.9: 1.25 (1.09-1.44); 35-39.9: 1.04 (0.86-1.25); >=40: 1.13 (0.…","""WEAKENED""","""weakened for 30-34.9"""
"""Part 2""","""3. Negative control: smoking""","""accident-death RR excludes 1 and overlaps the main RR -> weakened""","""any: 1.81 (1.57-2.08)""","""WEAKENED""","""weakened for any"""


**Result.** The method checks pass. Check 1 passes for every exposure class for infant death. Check 2 switched 5 rows to random effects without changing any conclusion. **Check 3 weakens the claims for smoking, short intervals and BMI 30–34.9**; hypertension, diabetes and severe obesity pass, but with few accident deaths. Check 5 was applied: imputation changes no RR by more than 1.2%, so no conclusion changes.

## 5b. Extra research (post-hoc, `06_extra_research`)

**What.** The checks of the extra research. They were added after the main results were known, but each rule was written in the plan before its run (change log 25 Nov 2026). They are listed apart (part "Extra") and do not change the counts of the pre-set checks.

In [6]:
ext = pl.read_csv(R / "extra_external_bmi.csv")
add("Extra", "E6. BMI births and rates vs NVSR 69(9)", "births within 0.5%, rates within 5% per BMI class",
    f"births max diff {ext['births_diff_pct'].abs().max():.2f}%, rates max diff {ext['rate_diff_pct'].abs().max():.1f}%",
    "PASS" if (ext["result"] == "PASS").all() else "FAIL")
for r in pl.read_csv(R / "extra_cause_predictions.csv").iter_rows(named=True):
    add("Extra", f"E1. {r['exposure']}: RR for {r['cause']} > other causes", "ratio of RRs > 1 and 95% CI excludes 1",
        f"ratio {r['ratio']:.2f} ({r['ratio_lo']:.2f}-{r['ratio_hi']:.2f})", "PASS" if r["prediction"] == "HOLDS" else "FAIL")
for r in pl.read_csv(R / "extra_bias_analysis.csv").iter_rows(named=True):
    add("Extra", f"E2. Recording bias: {r['exposure']}, {r['scenario']}", "FRAGILE if the 95% simulation interval includes 1",
        f"RR {r['rr_observed']:.2f} -> {r['total_median']:.2f} ({r['total_lo']:.2f}-{r['total_hi']:.2f})",
        "PASS" if r["verdict"] == "ROBUST" else "FRAGILE", r["source"])
for r in pl.read_csv(R / "extra_stillbirth.csv").iter_rows(named=True):
    add("Extra", f"E3. Stillbirth selection: {r['exposure']} {r['label']}", "composite RR / live-birth RR within 0.80-1.25",
        f"live {r['rr_live']:.2f}, fetal or infant {r['rr_composite']:.2f} (ratio {r['composite_over_live']:.2f})",
        "PASS" if r["verdict"] == "NOT BIASED" else "BIASED")
med = pl.read_csv(R / "extra_mediation.csv")
for name, rule, ok in (("hypertension", "proportion mediated: lower CI limit > 0.5", lambda r: r["prop_mediated_lo"] > 0.5),
                       ("smoking", "proportion mediated: upper CI limit < 0.5", lambda r: r["prop_mediated_hi"] < 0.5)):
    r = med.filter(pl.col("exposure") == name).row(0, named=True)
    add("Extra", f"E4. Mediation through gestational age: {name}", rule,
        f"{r['prop_mediated']:.2f} ({r['prop_mediated_lo']:.2f}-{r['prop_mediated_hi']:.2f})", "PASS" if ok(r) else "FAIL")
dca = pl.read_csv(R / "extra_decision_curves.csv")
for m, d in dca.group_by("model", maintain_order=True):
    use = d.filter(pl.col("useful"))["threshold"].to_list()
    add("Extra", f"E5. Decision curve: {m[0]}", "net benefit - max(treat all, 0): 95% CI above 0",
        f"useful at {len(use)} of {d.height} thresholds" + (f" ({100 * min(use):.2g}-{100 * max(use):.2g}%)" if use else ""), "INFO")
pl.DataFrame([c for c in checks if c["part"] == "Extra"]).select("check", "result", "status")

check,result,status
str,str,str
"""E6. BMI births and rates vs NVSR 69(9)""","""births max diff 0.00%, rates max diff 1.1%""","""PASS"""
"""E1. hypertension: RR for preterm / low birth weight > other causes""","""ratio 0.95 (0.88-1.03)""","""FAIL"""
"""E1. diabetes: RR for congenital anomalies > other causes""","""ratio 1.58 (1.45-1.71)""","""PASS"""
"""E1. smoking: RR for SUID > other causes""","""ratio 1.80 (1.73-1.86)""","""PASS"""
"""E2. Recording bias: hypertension, non-differential""","""RR 1.50 -> 1.75 (1.59-2.14)""","""PASS"""
"""E2. Recording bias: hypertension, differential (Se deaths +0.10)""","""RR 1.50 -> 1.36 (1.18-1.66)""","""PASS"""
"""E2. Recording bias: diabetes, non-differential""","""RR 1.84 -> 1.95 (1.81-2.26)""","""PASS"""
"""E2. Recording bias: diabetes, differential (Se deaths +0.10)""","""RR 1.84 -> 1.70 (1.52-1.99)""","""PASS"""
"""E2. Recording bias: smoking, non-differential""","""RR 1.68 -> 2.18 (1.97-2.57)""","""PASS"""


**Result.** 25 extra rows: 18 PASS, 1 FAIL, 1 BIASED, 5 INFO.
- **FAIL:** the hypertension cause-of-death prediction (the P07 group is not raised more than other causes). The mediation analysis suggests that most of the hypertension effect runs through earlier delivery (an upper bound; `06_extra_research` sections 7 and 9).
- **BIASED:** diabetes. With stillbirths included, its RR rises from 1.84 to 2.48, so the live-birth RR under-states its harm.
- The recording-bias analysis finds every RR robust; the decision curves show that the Part 1 models are useful at all thresholds from 0.25% to 5% (postneonatal: only up to 1–1.5%).

## 5c. Post-hoc audit (25 Nov 2026, GitHub issues #1–#5)

**What.** Sensitivity analyses added after an audit of the finished work (plan, change log 25 Nov 2026). They are listed apart (part "Audit") and change no pre-set status. Sources: `03_prediction` section 10, `04_association` section 19, `06_extra_research` section 9.

In [7]:
a1 = {r["model"]: r for r in pl.read_csv(R / "part1_audit.csv").iter_rows(named=True)}
s1, s2, dd = a1["d_hgb|B_strict"], a1["d_hgb|B_strict2"], a1["difference B_strict - B_strict2"]
add("Audit", "#4. Strict-2 model (also without gestational diabetes, gestational hypertension, WIC)",
    "not pre-set; reported next to the strict model",
    f"PR-AUC {s2['pr_auc']:.4f} ({s2['pr_auc_lo']:.4f}-{s2['pr_auc_hi']:.4f}) vs strict {s1['pr_auc']:.4f}; "
    f"difference {dd['pr_auc']:.4f} ({dd['pr_auc_lo']:.4f}-{dd['pr_auc_hi']:.4f})", "INFO")
tg = pl.read_csv(R / "part1_audit_tuning.csv")
old_best = tg.filter(~pl.col("new"))["tune_pr_auc"][0]
new_best = tg.filter(pl.col("new"))["tune_pr_auc"].max()
add("Audit", "#5. Wider boosting grid for Set B (chosen on 2021)", "tested on 2022-2023 only if a new setting wins on 2021",
    f"2021 PR-AUC old best {old_best:.4f}, best new {new_best:.4f}"
    + (f"; test {a1['d_hgb|B_retuned']['pr_auc']:.4f} vs {a1['d_hgb|B']['pr_auc']:.4f}" if "d_hgb|B_retuned" in a1 else "; not tested"),
    "INFO")
nca = pl.read_csv(R / "part2_audit_negative_control.csv").filter(pl.col("exposure") == "smoking")
add("Audit", "#2. Smoking negative control without fire / home hazards", "same rule as check 3",
    "; ".join(f"{r['outcome']}: {r['rr_accident']:.2f} ({r['lo']:.2f}-{r['hi']:.2f})" for r in nca.iter_rows(named=True)),
    "WEAKENED" if (nca["check3"] == "WEAKENED").all() else "PASS", "Fire explains a small part; the signal remains.")
gaa = pl.read_csv(R / "part2_audit_gestation.csv")
iv_max = gaa.filter(pl.col("analysis") == "interval_oe")["change_pct"].abs().max()
co_max = gaa.filter(pl.col("analysis") == "collider_ga_oe")["change_pct"].abs().max()
add("Audit", "#3. Obstetric-estimate gestational age (interval, collider demonstration)", "robust if no RR changes by > 5%",
    f"interval max change {iv_max:.1f}%; collider max change {co_max:.1f}%",
    "INFO", "Interval robust; collider RRs move further (hypertension 0.90 -> 0.84), same conclusion.")
mo = pl.read_csv(R / "extra_mediation_oe.csv")
add("Audit", "#1/#3. Mediation with obstetric-estimate gestational age (upper bounds)", "pre-set expectations judged again",
    "; ".join(f"{r['exposure']} {r['prop_mediated']:.2f} ({r['prop_mediated_lo']:.2f}-{r['prop_mediated_hi']:.2f})"
              for r in mo.iter_rows(named=True)),
    "INFO", "Proportions mediated are upper bounds: the collider check shows unmeasured gestational-age-death confounding.")
pl.DataFrame([c for c in checks if c["part"] == "Audit"]).select("check", "result", "status")

check,result,status
str,str,str
"""#4. Strict-2 model (also without gestational diabetes, gestational hypertension, WIC)""","""PR-AUC 0.0275 (0.0263-0.0288) vs strict 0.0300; difference 0.0025 (0.0021-0.0029)""","""INFO"""
"""#5. Wider boosting grid for Set B (chosen on 2021)""","""2021 PR-AUC old best 0.0370, best new 0.0372; test 0.0382 vs 0.0380""","""INFO"""
"""#2. Smoking negative control without fire / home hazards""","""all accidents (pre-set): 1.81 (1.57-2.08); without fire (149): 1.75 (1.51-2.03); without home hazards (148-150): 1.59 (1…","""WEAKENED"""
"""#3. Obstetric-estimate gestational age (interval, collider demonstration)""","""interval max change 1.1%; collider max change 10.6%""","""INFO"""
"""#1/#3. Mediation with obstetric-estimate gestational age (upper bounds)""","""hypertension 1.16 (1.10-1.22); diabetes 0.88 (0.85-0.93); smoking 0.36 (0.34-0.37)""","""INFO"""


## 6. The full checks table

**What.** All checks in one table, saved for `main.ipynb` and for the thesis appendix.

In [8]:
table = pl.DataFrame(checks)
table.write_csv(R / "checks.csv")
print(table.group_by("status").len().sort("status").rows())
table.select("part", "check", "result", "status")

[('APPLIED', 2), ('BIASED', 1), ('FAIL', 3), ('INFO', 14), ('PASS', 26), ('WEAKENED', 4)]


part,check,result,status
str,str,str,str
"""Step 1""","""Record counts vs user guides""","""births 7/8, deaths 8/8""","""FAIL"""
"""Step 1""","""CDC Documentation Table 2 reproduced""","""exact in 6 cohorts (2018-2023); off in [2016, 2017]""","""FAIL"""
"""Part 1""","""1. Best model beats age + race model""","""0.0293 (0.0277 to 0.0308)""","""PASS"""
"""Part 1""","""2. Calibration slope""","""0.942 (0.933 to 0.950); O/E 1.03""","""PASS"""
"""Part 1""","""3. Single-input leakage""","""highest feduc 0.626""","""PASS"""
"""Part 1""","""Post-hoc: model without timing items""","""Set B PR-AUC 0.0300 (0.0288-0.0314) vs 0.0380""","""INFO"""
"""Part 2""","""Model code vs statsmodels""","""beta 1.3e-12, SE 4.7e-12, RD 4.3e-12""","""PASS"""
"""Part 2""","""DAG adjustment sets (back-door)""","""5/5 valid, 5/5 invalid with GA""","""PASS"""
"""Part 2""","""1. Direction in >= 7 of 8 cohorts (infant death)""","""13/13 exposure classes""","""PASS"""


## Summary

- **Main finding.** Every pre-set check was run, and the results are reported as the rules say, including the failures.
- **PASS:** Part 1 checks 1–3, the model code, the DAG sets, Part 2 direction (check 1), and the negative control for hypertension and diabetes (low power).
- **APPLIED:** random effects where I² > 50% (check 2) and multiple imputation (check 5); neither changes a conclusion.
- **FAIL:** the record-count rule (births in 2020: a typo in the 2020 guide) and the strict CDC Table 2 rule in 2016 and 2017. Both are caused by CDC's own documents, not by our data.
- **WEAKENED:** the negative control for smoking, short intervals and BMI 30–34.9. These associations probably include unmeasured social confounding.
- **Extra research (post-hoc, `06_extra_research`):** 18 PASS, 1 FAIL (hypertension cause-of-death prediction), 1 BIASED (diabetes, live-birth selection), 5 INFO.
- **Audit (post-hoc, part "Audit", issues #1–#5):** strict-2 PR-AUC 0.0275; the wider grid adds +0.0002; the smoking negative control stays WEAKENED without fire, choking and poisoning (1.59); the interval is robust to the gestational-age measure; mediation proportions are upper bounds.
- **Open issues.** None in the analysis. The supervisor should review the DAGs.
- **Next.** Write the thesis text; `main.ipynb` shows all chapters.